# CyberXAI — NSL-KDD Data Preparation & Model Training

**Final Year Project · Intrusion Detection Model**

This notebook replaces local VS Code data prep. It:
1. Loads NSL-KDD train/test files
2. Prepares 4 features (`duration`, `src_bytes`, `dst_bytes`, `count`)
3. Trains Random Forest (same as `backend/train_model.py`)
4. Exports files for your CyberXAI app

### Files you need
Download from [NSL-KDD dataset page](https://www.unb.ca/cic/datasets/nsl.html):
- `KDDTrain+.txt`
- `KDDTest+.txt`

### After training
Download the zip and copy these into your project `models/` folder:
- `nslkdd_4f_rf_model.joblib`
- `nslkdd_4f_features.json`
- `nslkdd_4f_sample_input.csv`

## 1. Install dependencies (Colab usually has these)

In [ ]:
!pip install -q pandas scikit-learn joblib

## 2. Choose data source

Run **ONE** of the next two cells:
- **Option A** — Upload files directly (easiest)
- **Option B** — Use Google Drive (if files are already in Drive)

### Option A — Upload NSL-KDD files

In [ ]:
from google.colab import files
import shutil
from pathlib import Path

DATA_DIR = Path("/content/data/NSL-KDD")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Upload KDDTrain+.txt and KDDTest+.txt (select both files)")
uploaded = files.upload()

for name in uploaded:
    dest = DATA_DIR / name
    shutil.move(name, dest)
    print(f"Saved: {dest}")

TRAIN_FILE = DATA_DIR / "KDDTrain+.txt"
TEST_FILE = DATA_DIR / "KDDTest+.txt"

print("\nTrain exists:", TRAIN_FILE.exists())
print("Test exists:", TEST_FILE.exists())

### Option B — Google Drive (skip if you used Option A)

In [ ]:
from google.colab import drive
from pathlib import Path

# Change this path to where your NSL-KDD files live in Drive
DRIVE_DATA_PATH = "/content/drive/MyDrive/FYP/NSL-KDD"

drive.mount("/content/drive")

DATA_DIR = Path("/content/data/NSL-KDD")
DATA_DIR.mkdir(parents=True, exist_ok=True)

source_dir = Path(DRIVE_DATA_PATH)
TRAIN_FILE = source_dir / "KDDTrain+.txt"
TEST_FILE = source_dir / "KDDTest+.txt"

print("Train exists:", TRAIN_FILE.exists())
print("Test exists:", TEST_FILE.exists())

if not TRAIN_FILE.exists() or not TEST_FILE.exists():
    raise FileNotFoundError(
        f"Files not found in {source_dir}. "
        "Update DRIVE_DATA_PATH or use Option A upload."
    )

## 3. Load & prepare data

In [ ]:
import pandas as pd
from pathlib import Path

COLUMNS = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes", "land",
    "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in",
    "num_compromised", "root_shell", "su_attempted", "num_root", "num_file_creations",
    "num_shells", "num_access_files", "num_outbound_cmds", "is_host_login",
    "is_guest_login", "count", "srv_count", "serror_rate", "srv_serror_rate",
    "rerror_rate", "srv_rerror_rate", "same_srv_rate", "diff_srv_rate",
    "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
    "dst_host_same_srv_rate", "dst_host_diff_srv_rate", "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate", "dst_host_serror_rate", "dst_host_srv_serror_rate",
    "dst_host_rerror_rate", "dst_host_srv_rerror_rate", "attack", "level",
]

SELECTED_FEATURES = ["duration", "src_bytes", "dst_bytes", "count"]

if "TRAIN_FILE" not in globals():
    TRAIN_FILE = Path("/content/data/NSL-KDD/KDDTrain+.txt")
    TEST_FILE = Path("/content/data/NSL-KDD/KDDTest+.txt")

if not Path(TRAIN_FILE).exists():
    raise FileNotFoundError(f"Training file not found: {TRAIN_FILE}")
if not Path(TEST_FILE).exists():
    raise FileNotFoundError(f"Test file not found: {TEST_FILE}")

print("Loading NSL-KDD files...")
train_df = pd.read_csv(TRAIN_FILE, names=COLUMNS)
test_df = pd.read_csv(TEST_FILE, names=COLUMNS)

X_train = train_df[SELECTED_FEATURES].copy()
X_test = test_df[SELECTED_FEATURES].copy()

# Binary target: 0 = Normal, 1 = Attack
y_train = (train_df["attack"] != "normal").astype(int)
y_test = (test_df["attack"] != "normal").astype(int)

print(f"Train shape: {X_train.shape}")
print(f"Test shape:  {X_test.shape}")
print(f"Features:    {SELECTED_FEATURES}")
print(f"Train attacks: {y_train.sum()} / {len(y_train)}")
print(f"Test attacks:  {y_test.sum()} / {len(y_test)}")

display(X_train.describe())

## 4. Train Random Forest model

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import json
import joblib
from pathlib import Path

MODELS_DIR = Path("/content/models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
)

print("Training model...")
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("\n--- NSL-KDD 4-Feature Model Results ---")
print(f"Accuracy: {accuracy:.4f}")
print(classification_report(y_test, y_pred, target_names=["Normal", "Attack"]))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

## 5. Save artifacts & download for CyberXAI

In [ ]:
import shutil
from google.colab import files

model_path = MODELS_DIR / "nslkdd_4f_rf_model.joblib"
features_path = MODELS_DIR / "nslkdd_4f_features.json"
sample_path = MODELS_DIR / "nslkdd_4f_sample_input.csv"

joblib.dump(model, model_path)

with open(features_path, "w") as f:
    json.dump(SELECTED_FEATURES, f)

sample_df = X_test.head(5).copy()
sample_df["actual_label"] = y_test.head(5).values
sample_df.to_csv(sample_path, index=False)

print("Saved files:")
print(f"- {model_path}")
print(f"- {features_path}")
print(f"- {sample_path}")

zip_path = "/content/cyberxai_model_artifacts.zip"
shutil.make_archive("/content/cyberxai_model_artifacts", "zip", MODELS_DIR)

print("\nDownload zip and copy files into your project models/ folder:")
files.download(zip_path)

## 6. Quick test (same as backend /predict logic)

In [ ]:
sample = {
    "duration": 12,
    "src_bytes": 7000,
    "dst_bytes": 4000,
    "count": 30,
}

input_df = pd.DataFrame([sample])
pred = model.predict(input_df)[0]
proba = model.predict_proba(input_df)[0]

label = "Attack" if pred == 1 else "Normal"
confidence = proba[pred]

print(f"Prediction: {label}")
print(f"Confidence: {confidence:.4f}")